# iSeg-2017 — U-Net 2.5D frugal

Segmentation LCR / substance grise / substance blanche sur IRM T1-T2 de nourrissons de 6 mois.

Ce notebook **orchestre** seulement : toute la logique vit dans le paquet `iseg/`, versionné dans git.

**Avant de lancer** : Exécution → Modifier le type d'exécution → GPU T4.

## 1. Code et dépendances

In [ ]:
# Option A : depuis GitHub (remplacer par ton dépôt)
# !git clone https://github.com/<utilisateur>/iSeg-2017.git repo

# Option B : depuis Drive, si tu y as poussé le dossier du projet
from google.colab import drive
drive.mount('/content/drive')
!cp -r "/content/drive/MyDrive/iSeg-2017" /content/repo

%cd /content/repo
!pip -q install nibabel onnx onnxruntime onnxscript

import torch
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN — passer en T4')

## 2. Prétraitement

Normalisation z-score dans le masque cérébral, recadrage 144×144, mise en cache `.npz`.
Le cache complet pèse ~26 Mo : on peut le garder sur Drive et sauter cette étape ensuite.

In [ ]:
from iseg.data import build_cache
build_cache('iSeg-2017-Training', 'cache', range(1, 11))
!du -sh cache

## 3. Entraînement — modèle de référence

Validation croisée 5 blocs, découpage **par sujet**.
Compter ~10 min par bloc sur T4, soit ~50 min pour la référence complète.

In [ ]:
!python -m iseg.train --variant standard --modalities t1t2 --context 5 \
    --epochs 60 --batch-size 16 --eval-every 5 --out runs

## 4. Variantes frugales

Même protocole, modèles plus petits. C'est ce qui donne la courbe Dice vs taille.

In [ ]:
!python -m iseg.train --variant separable --epochs 60 --out runs
!python -m iseg.train --variant tiny      --epochs 60 --out runs

## 5. Ablations

Deux questions que l'exploration a soulevées :

- **Le T2 sert-il vraiment ?** Le ratio de Fisher GM/WM vaut 0,269 pour T1 seul et 0,001 pour T2 seul ;
  la meilleure combinaison linéaire ne gagne que 10 %. Si le Dice ne bouge pas sans T2, on divise
  par deux le prétraitement à l'inférence — excellent résultat de frugalité.
- **Combien de coupes de contexte ?** Puisque l'intensité seule ne sépare pas GM et WM,
  le contexte spatial est la seule ressource disponible.

In [ ]:
!python -m iseg.train --variant standard --modalities t1 --epochs 60 --out runs   # T1 seul
!python -m iseg.train --variant standard --context 1 --epochs 60 --out runs       # 2D pur
!python -m iseg.train --variant standard --context 7 --epochs 60 --out runs       # contexte élargi

## 6. Export ONNX et mesure de frugalité

In [ ]:
for variant in ['standard', 'separable', 'tiny']:
    !python -m iseg.export --checkpoint runs/{variant}_fold0.pt --cache cache \
        --calib-subjects 1 2 --runs 50 --threads 1 --out export

## 7. Tableau récapitulatif et courbe Dice vs frugalité

In [ ]:
import json, glob
import matplotlib.pyplot as plt

rows = [json.load(open(f)) for f in sorted(glob.glob('export/*_frugalite.json'))]

print(f"{'variante':<12}{'params':>10}{'Mo int8':>10}{'MMACs':>9}{'ms/coupe':>10}{'Dice':>8}")
for r in rows:
    q = r.get('int8', r['fp32'])
    print(f"{r['variant']:<12}{r['params']:>10,}{q['mb']:>10.2f}"
          f"{r['macs_per_slice']/1e6:>9.1f}{q['ms_mean']:>10.2f}{r['dice']['dice_mean']:>8.4f}")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for r in rows:
    ax.scatter(r['params'], r['dice']['dice_mean'], s=90)
    ax.annotate(r['variant'], (r['params'], r['dice']['dice_mean']),
                textcoords='offset points', xytext=(8, -3))
ax.set_xscale('log')
ax.set_xlabel('nombre de paramètres (échelle log)')
ax.set_ylabel('Dice moyen (LCR, GM, WM)')
ax.set_title('Compromis performance / frugalité')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('figures/frugalite.png', dpi=140)

## 8. Récupérer les résultats

Les poids `.pt`, les `.onnx` et les journaux `.json` sont copiés sur Drive.
Le `.onnx` quantifié est le fichier qui alimentera la page web de démonstration.

In [ ]:
!mkdir -p "/content/drive/MyDrive/iSeg-2017-resultats"
!cp -r runs export figures "/content/drive/MyDrive/iSeg-2017-resultats/"
!ls -la "/content/drive/MyDrive/iSeg-2017-resultats/export"